# Les inn Lånekassen-data fra Omsetjingsminne 
Se [https://www.nb.no/sprakbanken/ressurskatalog/oai-nb-no-sbr-78/](https://www.nb.no/sprakbanken/ressurskatalog/oai-nb-no-sbr-78/)

In [ ]:
from collections import defaultdict
import pandas as pd
import xml.etree.ElementTree as ET

tree = ET.parse("lanekassen.no.csv-nb-nn.deduped.tmx")
root = tree.getroot()

data_df = defaultdict(list)

body = root.find("body")
tus = body.findall("tu")
for tu in tus:
    if tu.attrib["datatype"] == "Text":
        for e in tu:
            if e.tag == "prop":
                if "score" in e.attrib["type"]:
                    data_df["_".join(e.attrib["type"].split("-"))].append(e.text)
            elif e.tag == "tuv":
                assert len(e.attrib) == 1
                lang = e.attrib["{http://www.w3.org/XML/1998/namespace}lang"]

                urls = [x.text for x in e if x.tag == "prop"]
                texts = [x.text for x in e if x.tag == "seg"]
                assert len(texts) == 1

                data_df[f"{lang}_segment"].append(texts[0])
                data_df[f"{lang}_urls"].append(urls)

df = pd.DataFrame(data_df)
df

# Match opp til doc_hash via url

In [ ]:
lånekassen_data = pd.read_json("lånekassen_data.json")
lånekassen_data

In [ ]:
len(set(lånekassen_data.url)), len(set(lånekassen_data.doc_hash))

In [ ]:
# Max antall mulige setningspar
nob_antall_setninger = lånekassen_data[lånekassen_data.lang == "nob"].fulltext.apply(len).sum()
nno_antall_setninger = lånekassen_data[lånekassen_data.lang == "nno"].fulltext.apply(len).sum()
min((nno_antall_setninger, nob_antall_setninger))

In [ ]:
url_to_doc_hash = {e.url: e.doc_hash for e in lånekassen_data.itertuples()}

In [ ]:
df["nb_doc_hash"] = df.nb_urls.apply(lambda x: [url_to_doc_hash[url] for url in x if url in url_to_doc_hash])
df["nn_doc_hash"] = df.nn_urls.apply(lambda x: [url_to_doc_hash[url] for url in x if url in url_to_doc_hash])

In [ ]:
har_nn_hash = df[df.nn_doc_hash.apply(len)>=1]
har_nb_hash = df[df.nb_doc_hash.apply(len)>=1]
har_hash = df[(df.nn_doc_hash.apply(len)>=1) & (df.nb_doc_hash.apply(len)>=1)]

len(har_nb_hash), len(har_nn_hash), len(har_hash)

In [ ]:
print(f"""
Av {len(df)} setningspar er det 
{len(har_nb_hash)} bokmålsetninger med url som mapper til en hash (og {len(df)-len(har_nb_hash)} som mangler)
{len(har_nn_hash)} nynorsksetninger med url som mapper til en hash (og {len(df)-len(har_nn_hash)} som mangler)
{len(har_hash)} setningspar med url som mapper til hash for begge språk (og {len(df)-len(har_hash)} som mangler)
""")

# Sammenlikn med våre matches

In [ ]:
from pathlib import Path
import pandas as pd

p = Path("output")


for e_ in p.iterdir():
    if e_.is_dir():
        for e in e_.iterdir():
            if "flat" in e.name:
                df_ = pd.read_csv(e)
                antall_par_oss = len(df_)
                df_ = df_.merge(df, left_on="nn_text", right_on="nn_segment")
                same_pairs = df_[df_.bm_text == df_.nb_segment]

                print(f"Med {e_.name}/{e.name} fant vi {antall_par_oss} par (mot {len(df)} i omsetjingsminne) \
                \nAv disse var det {len(df_)} av våre nynorsktekster som fantes ordlikt i omsetjingsminnet ({round(len(df_)/antall_par_oss*100, 2)}% av våre par og {round(len(df_)/len(df)*100, 2)}% av para i omsetjingsminnet) \
                \nAv de {len(df_)} ordlike nynorsktekstene, hadde vi samme bokmålstekst som omsetjingsminnet i {len(same_pairs)} tilfeller \
                \nDette er {round(len(same_pairs)/len(df_)*100, 2)}% av parene\n\n")
